# Detailed Walkthrough of the LopezAcosta2019 Pipeline
In this notebook, we examine the LopezAcosta2019 workflow in detail. First, we add the IFT and Images libraries.

In [ ]:
using Pkg
Pkg.add("IceFloeTracker")
Pkg.add("Images")

Next we load the IFT and Images packages, and import the internal functions from the LopezAcosta2019 module.

In [ ]:
using IceFloeTracker
using Images
import IceFloeTracker.LopezAcosta2019:
    discriminate_ice_water,
    clean_binary_floes,
    segB_binarize,
    segmented_ice_cloudmasking,
    reconstruct_and_mask,
    morph_split_floes,
    watershed_ice_floes,
    watershed_product,
    IceDetectionLopezAcosta2019

## 1. Load data
For this example, we'll use a subset of a large image from the western Arctic Ocean. We need the true color, false color, and land mask images. We can use `mosaicview` to view the images side-by-side. The true color image uses MODIS bands 1, 4, and 3 (red, green, blue) and approximates the scene perceived by a human eye. The false color image uses MODIS bands 7, 2, and 1 (near infrared, shortwave infrared, and red).

In [ ]:
HOME = "../.."
test_region = (1:2707, 1:4458)

truecolor_image = float64.(RGB.(load(joinpath(HOME, "test/test_inputs/beaufort-chukchi-seas_truecolor.2020162.aqua.250m.tiff"))[test_region...]))
falsecolor_image = float64.(RGB.(load(joinpath(HOME, "test/test_inputs/beaufort-chukchi-seas_falsecolor.2020162.aqua.250m.tiff"))[test_region...]))
land_mask_img = load(joinpath(HOME, "test/test_inputs/landmask.tiff"))[test_region...]

mosaicview(truecolor_image, falsecolor_image, land_mask_img, nrow=1)

## 2. Landmask generation
As seen above, the landmask is an RGB image with lots of details near the coast. The `create_landmask` function converts this image into two: a bitmatrix version of the original mask, with 1s meaning land, and a `coastal_buffer` with a smoothed and expanded mask. This `coastal_buffer` can help mask ambiguous near-coast variables, such as bright pixels from sandbars or river outflow. The coastal buffer is made with a structuring element (SE). Here, we use the built-in box SE from ImageMorphology.

In [ ]:
# 2. Create landmask
@time coastal_buffer, land_mask = create_landmask(land_mask_img, strel_box((51,51)));

# Display the land masks. Using "Gray" tells Julia to interpret the array of 0's and 1's as an image.
mosaicview(Gray.(land_mask), Gray.(coastal_buffer), nrow=1)

## 3. Cloud mask
The false color image uses the near infrared and shortwave infrared channels to make most clouds brighter than sea ice, snow, and water. In the LopezAcosta2019 routine, only the brightest clouds are masked. With this method, additional checks for cloud filtering are needed after processing to remove false positives, but floes under thin clouds can often be detected.

The parameters shown below are the LopezAcostaCloudMask default parameters, and are just shown here so that you can experiment with other values more easily.

In [ ]:
# set parameters for cloudmask
prelim_threshold = Float64(110 / 255)
band_7_threshold = Float64(200 / 255)
band_2_threshold = Float64(190 / 255)
ratio_lower = 0.0
ratio_offset = 0.0
ratio_upper = 0.75

# Create cloudmask from reflectance image
@time cloudmask = create_cloudmask(falsecolor_image,
        LopezAcostaCloudMask(prelim_threshold,
                             band_7_threshold,
                             band_2_threshold,
                             ratio_lower,
                             ratio_offset,
                             ratio_upper));

# This cloudmask is only valid over ocean and ice, so we should mask any land pixels
apply_landmask!(cloudmask, land_mask)

# cloud is white
mosaicview(falsecolor_image, Gray.(cloudmask), nrow=1)

## 3. Preprocessing
The IFT includes multiple functions for image preprocessing. Additional functions from Images (e.g. ImageFiltering, ImageContrastAdjustment) can also be used. For the LopezAcosta2019 module, the preprocessing includes nonlinear diffusion, adaptive histogram equalization, and sharpening via the unsharp masking technique. With these techniques, we create a grayscale image with enhanced contrast at ice floe edges and reduced contrast within ice floes, and we even out the lighting across the image. 

In [ ]:
# 1. Nonlinear diffusion using the Perona-Malik algorithm
pmd = PeronaMalikDiffusion(0.1, 0.1, 5, "exponential")
@time truecolor_diffused = nonlinear_diffusion(truecolor_image, pmd)

# 2. Adaptive histogram equalization
truecolor_equalized = adjust_histogram(truecolor_diffused,
                      AdaptiveEqualization(nbins=256, rblocks=8, cblocks=10, clip=0.8))

# 3. Unsharp Mask and Convert to Grayscale
truecolor_sharpened = Gray.(unsharp_mask(truecolor_equalized))

# Apply masks
preprocessed_image = apply_landmask(truecolor_sharpened, land_mask)
apply_cloudmask!(preprocessed_image, cloudmask);

To see the effect of the preprocessing steps, we zoom in to a region with sea ice floes, and show (clockwise from top left) the original truecolor image, the result of nonlinear diffusion, the adaptive histogram equalization, and the sharpened grayscale image.

In [ ]:
zoom_region = (1000:2000, 2000:3000)
mosaicview(truecolor_image[zoom_region...],
           truecolor_diffused[zoom_region...],
           truecolor_equalized[zoom_region...],
           preprocessed_image[zoom_region...], nrow=2, rowmajor=true)

## 4. Segmentation
The segmentation workflows each involve methods to enhance contrast between floes and decrease variance within floes, a binarization method, and then a floe separation method. We illustrate the main methods here.

### 4.1 $k$-means binarization workflow
The preprocessed image is further processed using the `discriminate_ice_water` function. This function inverts the image, sets regions lower than a threshold to 0, and applies a threshold based on histogram properties. 

The k-means binarization method uses k-means to cluster the inverted image pixels by color. Leveraging the similarity of ice floes within an image, floes tend to be isolated well by the k-means clustering. To select a cluster containing floes, we use an IceDetectionAlgorithm to produce a binarized image with likely ice pixels. Then, the k-means clusters with the largest fraction of detected ice pixels is selected and set to 1, and all other pixels are set to background (0). 

Finally, the `clean_binary_floes` function applies morphological operations (branch, hbreak, area opening, hole filling) to clean up the resulting classified image.

In [ ]:
# 1. Enhance grayscale
@time ice_water_discrim = discriminate_ice_water(
    preprocessed_image, falsecolor_image, coastal_buffer, cloudmask)

# 2. Compute the k-means binarization
ice_detection_algorithm = IceDetectionLopezAcosta2019()
fc_masked = apply_landmask(falsecolor_image, coastal_buffer)
fc_masked .= apply_cloudmask(falsecolor_image, cloudmask)
kmeans_results = kmeans_binarization(ice_water_discrim, fc_masked; k=4, cluster_selection_algorithm=ice_detection_algorithm)

# 3. Cleanup
segmentation_A = clean_binary_floes(kmeans_results)

# Visualize segmentation A process
mosaicview(preprocessed_image[zoom_region...], ice_water_discrim[zoom_region...],
  Gray.(kmeans_results[zoom_region...]), Gray.(segmentation_A[zoom_region...]), nrow=2, rowmajor=true)

### 4.2 Global thresholds and gamma correction
In the second segmentation routine, we take a similar approach. Here, the grayscale image enhancement is brightening regions that are brighter than an initial threshold. The binarization function uses a ImageContrastAdjustment GammaCorrection algorithm to brighten floes and uses a threshold and morphological cleanup.

After creating the binarized image, the segmentation A and B results are joined by intersection. In this case, the Segmentation B result essentially sets all possible sea ice pixels to 1.

In [ ]:
prelim_binarized = preprocessed_image .> 0.4
brightened_gray = imbrighten(preprocessed_image, prelim_binarized, 1.3)
segmentation_B = segB_binarize(preprocessed_image, brightened_gray, cloudmask)

# Join segmentations results
ice_intersect = segmentation_A .* segmentation_B

mosaicview(
    Gray.(prelim_binarized[zoom_region...]),
    brightened_gray[zoom_region...], 
    Gray.(segmentation_B[zoom_region...]),
    Gray.(ice_intersect[zoom_region...]), 
    nrow=2, rowmajor=true
)

### 4.3 Identify common boundaries
We use a watershed transformation to find the common boundaries between the preliminary binarized mask and the merged segmentation A / B mask. 

In [ ]:
# Generate watersheds
@time watersheds_segB = map(watershed_ice_floes, [prelim_binarized, ice_intersect])
# 100.336801 seconds (3.02 M allocations: 267.685 GiB, 5.30% gc time, 0.95% compilation time)
watershed_intersect = watershed_product(watersheds_segB...);

## 5. Separate ice floes
The segmentation F routine repeats the segmentation with kmeans binraization after grayscale morphological operations, then uses a series of morphological operators to separate the floes from each other.

In [ ]:
morphed_grayscale = reconstruct_and_mask(
    preprocessed_image,
    watershed_intersect,
    ice_intersect
) 
# K-means binarization of reprocessed grayscale image
segF_binarized = kmeans_binarization(
    morphed_grayscale, RGB.(fc_masked);
    k=4,
    cluster_selection_algorithm=IceDetectionLopezAcosta2019(),
    ) .* .! watershed_intersect

# Split floes
final_floes = morph_split_floes(segF_binarized, cloudmask)
labels = label_components(final_floes)

# 14.314099 seconds (38.86 M allocations: 7.831 GiB, 6.88% gc time, 13.82% compilation time)
mosaicview(complement.(morphed_grayscale), Gray.(segF_binarized), Gray.(final_floes), nrow=1)

# 6. Visualize Results

In [ ]:
outlines = bwperim(final_floes)
overlay_img = RGB.(preprocessed_image)
overlay_img[outlines .> 0] .= RGB(1, 0, 0)
zoom2 = (1500:2500, 2000:3000)
mosaicview(preprocessed_image[zoom2...], overlay_img[zoom2...], nrow=1)

Some final notes:
* As discussed in Watkins et al. (2025) [(preprint)](https://essopenarchive.org/doi/full/10.22541/essoar.175503380.04848000) this method erodes the ice floes, so the true boundary is somewhat outside the floe boundary. If you wish to use the output to study floe sizes, the areas need to be adjusted by approximately 8 pixels on all sides.
* The method identifies _candidate floes_, not 100% definite floes. Tracking objects between images, and using shape-based and color-based filtering is necessary to remove false positives. 